Intermediate Deep Learning - Deep Reinforcement Learning
# Session 1: Foundations of Deep Reinforcement Learning - TD 1

## Part 1: Discovery of Gymnasium
---

### Task 1.0 ⚙️: Install packages

**Instructions:**

1. Install gymnasium[classic-control] which includes the Cartpole environment. Numpy is needed for Q table and matplotlib for data visualization.

In [1]:
# Check Python executable path
import sys
print(sys.executable)

# Install required packages
# !"{sys.executable}" -m pip install "gymnasium[classic-control]" numpy matplotlib

/Users/alivakof/Documents/Codex/python-data-science/.venv/bin/python


### Task 1.1 📝: Initialize a Cartpole environment

**Instructions:**
1. Initialize the Cartpole environment. Identify and describe the following:
    - **State**?
    - **Action**?
    - **Rewards**?
    - Differences between *terminated (done)* and *truncated*?  
    > You can find more information here: <https://gymnasium.farama.org/environments/classic_control/cart_pole/>.  
2. Complete the code to run a test for 1000 steps using random actions.

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

env = gym.make("CartPole-v1", render_mode="human")

# Reset environment to start a new episode
observation, info = env.reset()
# observation: what the agent can "see" - cart position, velocity, pole angle, etc.
# info: extra debugging information (usually not needed for basic learning)

print(f"Starting observation: {observation}")
# Example output: [ 0.01234567 -0.00987654  0.02345678  0.01456789]
# [cart_position, cart_velocity, pole_angle, pole_angular_velocity]

for _ in range(1000):
    action = env.action_space.sample()
    observation, reward, terminated, truncated, info = ...
    if terminated or truncated:
        observation, info = ...
env.close()

## Part 2: Design of a Q-Learning agent
---

### Task 2.1 📝: Initilize the class and hyperparameters

**Reminder 👨‍💻:**  the Q-learning update rule:
$$ Q'(s,a) = Q(s,a) + \alpha[r + \gamma \cdot \max_{a'}(Q(s',a')) - Q(s,a)]$$

**Instructions:**
1. Create a class called `QLearningAgent`
2. Define `__init__` method with necessary parameters:    
    

In [ ]:
class QLearningAgent:
    def __init__(self, ...):
        """Q-Learning agent for CartPole-v1"""

        # Q-Learning hyperparameters
        ...

        # Environment parameters
        ...


### Task 2.2 📝: Discretization of state

**Reminder 👨‍💻:** 
Tabular Q-learning keep track of all Q-value in a table -> It is not directly applicable on continous state / action spaces.  

Solution: Discretization of spaces, which means transfering continuous spaces to discrete ones.  

**Instructions:**
1. Design a method called `discretize_state` that takes `state` as parameter an return the converted discrete state indices.   
    - We assume a hyperparameter `n_bins` is chosen for the agent during initialization indicating number of bins to discretize each state.  
    - Modify the `__init__` method accordingly.  
2. Test your implementation using provided code. 

<details>
<summary>Hint 💡</summary>

1. Store `n_bins` as instance variable in `__init__`: `self.n_bins = n_bins`. 
2. Create a list called `self.state_bounds` inside `__init__`, which stores tuples representing `(min, max)` bound of each state dimension:  
    - Cart position: (-4.8, 4.8)
    - Cart velocity: *(-5, 5)*
    - Pole angle: (-0.418, 0.418) - this is approximately ±24 degrees in radians
    - Pole angular velocity: *(-5, 5)*
3. Initialize an empty list called `discrete`.  
4. Loop through each dimension.  
    - Clip the state value to be within bounds using. *(why?)*
    - Scale the clipped value to $[0, 1]$.
    - Convert the scaled value to bin index.
    - Append the bin index to the `discrete` list
5. Return `discrete`

</details>

<details>
<summary>Toolbox 🛠️</summary>

- `np.clip(x, min, max)`

</details>



In [ ]:
class QLearningAgent:
    def __init__(self, ...):
        """Q-Learning agent for CartPole-v1"""

        # Q-Learning hyperparameters
        ...

        # Environment parameters
        ...

        # State discretization parameters
        ...
        
    def discretize_state(self, state):
        """
        Discretize continuous state into discrete bins
        
        Args:
            state (array): Continuous state from the environment
            
        Returns:
            tuple: Discretized state as a tuple of bin indices
        """
        ...
        return ...
    
    

In [ ]:
# Test code
agent = QLearningAgent(n_bins=10)
test_state = [0.0, 0.5, 0.1, -0.2]
print("Discrete state:", agent.discretize_state(test_state))
# Should print something like: (4, 4, 5, 4)
test_state = [-4.8, -10, 0.418, 3]
print("Discrete state:", agent.discretize_state(test_state))
# Should print something like: (0, 0, 9, 7)

### Task 2.3 📝: Initialize the Q-table

**Instructions:**
1. Create `self.q_table` as a numpy array of zeros in `__init__`. What's its dimension?

<details>
<summary>Toolbox 🛠️</summary>

- `np.zeros((dims))`

</details>


In [ ]:
class QLearningAgent:
    def __init__(self, ...):
        """Q-Learning agent for CartPole-v1"""

        # Q-Learning hyperparameters
        ...

        # Environment parameters
        ...

        # State discretization parameters
        ...
        
        # Initialize Q-table with zeros
        ...
        
    def discretize_state(self, state):
        """
        Discretize continuous state into discrete bins
        
        Args:
            state (array): Continuous state from the environment
            
        Returns:
            tuple: Discretized state as a tuple of bin indices
        """
        ...
        return ...

### Task 2.4 📝: Action Selection

**Reminder 👨‍💻:**  
In Q-learning with $\epsilon$-greedy strategy, we have two way for action selection:  
- Select action randomly (exploration). 
- Select action that gives highest Q-value (exploitation). 

**Instructions:**
1. Create a method called `choose_action` that takes `state` as parameter
2. Generate a random number between 0 and 1
3. If the random number is less than $\epsilon$:
   - Return a random action
4. Otherwise:
   - Convert state to discrete indices
   - Access the Q-table at the discrete state
   - Return the action with highest Q-value

**Question:** What happens when epsilon = 1.0? When epsilon = 0.0?  

<details>
<summary>Toolbox 🛠️</summary>

- `np.random.random()`
- `np.random.randint(min, max)`
- `np.argmax(x)`

</details>

In [ ]:
def choose_action(self, state):
    """Epsilon-greedy action selection"""
    if ...:
        return ...  # Random action
    else:
        ...
        return ...

### Task 2.5 📝: Q-table Update

**Reminder 👨‍💻:**  the Q-learning update rule:
$$ Q'(s,a) = Q(s,a) + \alpha[r + \gamma \cdot \max_{a'}(Q(s',a')) - Q(s,a)]$$

**Instructions:**
1. Create a method called `update_q_table` with parameters:
   - `state`: Current state
   - `action`: Action taken
   - `reward`: Reward received
   - `next_state`: State after taking action
   - `done`: Boolean indicating if episode ended
2. Convert both states to discrete indices. 
3. Get the current Q-value from the Q-table.  
4. Calculate the target Q-value.  
    - If the episode id terminated (`done`)?
    - Else?
5. Update the Q-value. 



<details>
<summary>Toolbox 🛠️</summary>

- `np.max(x)`

</details>




In [ ]:
def update_q_table(self, state, action, reward, next_state, done):
    """Update Q-table using Q-learning update rule"""
    # Discretize states
    ...
    
    # Get current Q-value
    ...

    # Calculate target Q-value
    if done:
        ...
    else:
        ...

    # Update Q-value
    ...

### Task 2.6 (Optional)🌟: Add epsilon decay method

**Instructions:**
1. Create a method called `decay_epsilon` (no parameters besides self). 
2. Add instance variables in `__init__`: `self.epsilon_decay`, `self.epsilon_min`. 
3. Multiply `self.epsilon` by `self.epsilon_decay`. 
4. Use `max()` to ensure it doesn't go below `self.epsilon_min`. 
5. Update `self.epsilon` with this value. 

In [ ]:
def decay_epsilon(self):
    """Decay exploration rate"""
    ...

## Part 3: Training Loop
---

### Task 3.1 📝: Create the training function

**Instructions:**
1. Create a function called `train_agent` outside the agent class
2. Add parameters: `episodes` (default 1000) and `render` (default False)
3. Create the environment: `gym.make('CartPole-v1', render_mode='human' if render else None)`
4. Create an instance of `QLearningAgent`
5. Initialize an empty list `rewards_per_episode`

In [1]:
def train_agent(episodes=1000, render=False):
    """Train the Q-learning agent"""
    # Create environment and agent
    ...
    
    # Initialize list to store rewards per episode
    ...
    
    print("Starting training...")
    print("=" * 60)


### Task 3.2 📝: Implement the episode loop

**Instructions:**
1. Loop through each episode using `for episode in range(episodes):`
2. Reset the environment: `state, _ = env.reset()`
3. Initialize `total_reward` and `done`
4. Create a while loop that runs while `not (done or truncated)`

In [ ]:
for episode in range(episodes):

    # Reset environment
    ...

    # Initialize variables
    ...
    
    while not (done or truncated):
        # Steps will go here
        pass


### Task 3.3 📝: Implement the step loop

**Instructions:**
Inside the while loop:
1. Choose an action using `choose_action()`. 
2. Take the action using `env.step(action)`. 
3. Set `done = terminated`. 
4. Update the Q-table. 
5. Move to the next state. 
6. Accumulate reward. 

In [ ]:

while not (done or truncated):
    # Choose action
    ...
    
    # Take action in environment
    ...
    
    # Update Q-table
    ...
    
    # Move to next state
    ...


### Task 3.4 📝: Complete the training loop

**Instructions:**
After the while loop (but still inside the for loop):
1. (If implemented) Call `agent.decay_epsilon()`
2. Append `total_reward` to `rewards_per_episode`
3. Every 100 episodes, print progress:
   - Calculate average of last 100 rewards
   - Print episode number, average reward, and current epsilon
4. After all episodes, close environment and return agent and rewards

<details>
<summary>Toolbox 🛠️</summary>

- `np.meam(x)`

</details>

In [4]:
def train_agent(episodes=1000, render=False):
    """Train the Q-learning agent"""
    # Create environment and agent
    ...
    
    # Initialize list to store rewards per episode
    ...
    
    print("Starting training...")
    print("=" * 60)

    for episode in range(episodes):

        # Reset environment
        ...

        # Initialize variables
        ...
        
        while not (done or truncated):
            # Choose action
            ...
            
            # Take action in environment
            ...
            
            # Update Q-table
            ...
            
            # Move to next state
            ...
    
        # Decay exploration rate
        ...
                
        # Print progress every 100 episodes
        if (episode + 1) % 100 == 0:
            avg_reward = np.mean(rewards_per_episode[-100:])
            print(f"Episode {episode + 1}/{episodes}, "
                f"Avg Reward (last 100): {avg_reward:.2f}, "
                f"Epsilon: {agent.epsilon:.3f}")

    # Close environment
    ...

    print("=" * 60)
    print("Training complete!")
    return agent, rewards_per_episode

## Part 4: Visualization
---

### Task 4.1 📝: Create plotting function

**Instructions:**
1. Create a function called `plot_results` that takes `rewards` as parameter
2. Create a figure with 2 subplots side by side: `plt.subplots(1, 2, figsize=(12, 5))`
3. In the first subplot (axes[0]):
   - Plot raw rewards with transparency: `plt.plot(rewards, alpha=0.3)`
   - Add title, labels, and grid
4. In the second subplot (axes[1]):
   - Calculate moving average using `np.convolve()` with window size 100
   - Plot the moving average
   - Add title, labels, and grid
5. Call `plt.tight_layout()` and `plt.show()`

**What is moving average?** It smooths out noise by averaging over a window of episodes, making trends easier to see.

<details>
<summary>Toolbox 🛠️</summary>

- `np.convolve(x, np.ones(window)/window, mode='valid')` gives the moving average of `x` over specified window size.

</details>


In [ ]:
def plot_results(rewards):
    """
    Plot training results with raw rewards and moving average
    
    Args:
        rewards (list): List of rewards per episode
    """
    # Create figure with two subplots side by side
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Plot raw rewards in first subplot
    axes[0].plot(rewards, alpha=0.3)
    axes[0].set_title('Raw Rewards per Episode')
    axes[0].set_xlabel('Episode')
    axes[0].set_ylabel('Total Reward')
    axes[0].grid(True)
    
    # Calculate and plot moving average in second subplot
    window_size = 100
    moving_avg = ...
    axes[1].plot(moving_avg)
    axes[1].set_title(f'Moving Average (window={window_size})')
    axes[1].set_xlabel('Episode')
    axes[1].set_ylabel('Average Reward')
    axes[1].grid(True)
    
    # Adjust layout and display
    plt.tight_layout()
    plt.show()

## Part 5: Testing
---

### Task 5.1 📝: Create test function

**Instructions:**
1. Create `test_agent` function with parameters: `agent`, `episodes` (default 5)
2. Create environment with rendering: `gym.make('CartPole-v1', render_mode='human')`
3. For each test episode:
   - Reset environment
   - While not done or truncated:
     - Get discrete state
     - Choose best action (no exploration):
     - Take action and accumulate reward
   - Print reward for each episode
4. Close environment

In [ ]:
def test_agent(agent, episodes=5):
    """Test the trained agent"""
    env = gym.make('CartPole-v1', render_mode='human')
    
    print("\nTesting trained agent...")
    for episode in range(episodes):
        # Reset environment
        ...

        # Initialize variables
        total_reward = 0
        done = False
        
        while not done:
            # Choose best action (no exploration)
            ...
            
            # Take action in environment
            ...
        
        print(f"Test Episode {episode + 1}: Reward = {total_reward}")
    
    env.close()

## Part 6: Execution
---

### Task 6.1 🚀: Train and Evaluate the Agent

**Instructions:**
1. Run `train_agent` to train the agent
2. Plot training results using `plot_results`
3. Run `test_agent` to evaluate the trained agent
4. Interpret the results

In [ ]:
if __name__ == "__main__":
    print("Training Q-Learning Agent on CartPole-v1...")
    print("=" * 60)

    # Train the agent
    agent, rewards = train_agent(episodes=1000, render=False)

    print("\n" + "=" * 60)
    print(f"Training completed!")
    print(f"Final average reward (last 100 episodes): {np.mean(rewards[-100:]):.2f}")

    # Plot results
    plot_results(rewards)

    # Test the trained agent
    print("\nTesting trained agent...")
    test_agent(agent, episodes=5)